# M5 Evaluation Harness: Arabic Flood-Crisis Tweet Classification

This notebook scores the single-agent baseline on the frozen golden set (140 tweets, three seeds).

Sections 1 to 5 need **no API key and no API calls**: they read the saved runs in `m5/runs/` and score them.
Section 6 is optional and repeats the API runs; it is switched off by default.

## 1. Get the code

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "/content/arabic-flood-m4-baseline"
    if not os.path.exists(REPO):
        !git clone -q https://github.com/Maya-mohammed/arabic-flood-m4-baseline.git {REPO}
    %cd {REPO}
    !git pull -q
    !pip install -q -r requirements.txt pytest
print("Working folder:", os.getcwd())

## 2. Offline unit tests

In [ ]:
!python -m pytest -q tests

## 3. The golden set

`m5/golden_set_ids.csv` holds the 140 case IDs with their gold labels and flood event. It has no tweet text.
The cases were drawn from the reserved evaluation split (never used during development): 20 per Information Type, `random_state=2026`.

In [ ]:
import pandas as pd

golden = pd.read_csv("m5/golden_set_ids.csv", dtype=str)
print("Cases:", len(golden))
print(golden["joint_label"].value_counts().to_string(), "\n")
pd.crosstab(golden["Information Type"], golden["Event"], margins=True)

## 4. Score the baseline runs

Each file in `m5/runs/baseline/` is one run of `evaluate.py` over the 140 cases with one seed.
Cost uses the paid Standard prices Google lists for `gemini-3.5-flash-lite` ($0.30 per 1M input tokens, $2.50 per 1M output tokens).
The runs themselves used the free tier.

In [ ]:
!python score_runs.py --condition baseline \
    --runs m5/runs/baseline/seed_42.csv m5/runs/baseline/seed_123.csv m5/runs/baseline/seed_2026.csv \
    --out m5/results/baseline --price-input 0.30 --price-output 2.50

In [ ]:
pd.read_csv("m5/results/baseline/per_class_f1.csv").round(3)

In [ ]:
print("Information Type confusion matrix, all three seeds added (rows = gold, columns = prediction)")
pd.read_csv("m5/results/baseline/confusion_information_type_all_seeds.csv", index_col="gold")

In [ ]:
print("Relevance confusion matrix, all three seeds added")
pd.read_csv("m5/results/baseline/confusion_relevance_all_seeds.csv", index_col="gold")

## 5. Seed check (saved files only)

Two checks that need no API calls:
- how many cases got a different final pair across the three seeds;
- whether the pilot run (first five cases, seed 42, run in a separate session) matches the full seed 42 run at every stage.

In [ ]:
keys = ["pred_relevance", "pred_information_type"]
runs = {s: pd.read_csv(f"m5/runs/baseline/seed_{s}.csv", dtype=str).set_index("case_id") for s in (42, 123, 2026)}
changed = [cid for cid in runs[42].index
           if len({tuple(runs[s].loc[cid, keys]) for s in runs}) > 1]
print(f"Cases whose final pair differs across the three seeds: {len(changed)} of {len(runs[42])}")

stages = ["initial_relevance", "initial_information_type", "review_relevance",
          "review_information_type", "pred_relevance", "pred_information_type", "output_tokens"]
pilot = pd.read_csv("m5/runs/baseline_pilot/seed_42_pilot.csv", dtype=str).set_index("case_id")
same = (pilot[stages] == runs[42].loc[pilot.index, stages]).all(axis=1)
print(f"Pilot cases identical to the seed 42 run at every stage: {int(same.sum())} of {len(pilot)}")

## 6. Optional: repeat the API runs

Set `RUN_API = True` only to repeat the runs. It needs:
- a Gemini key in Colab Secrets named `GEMINI_API_KEY`;
- the corpus, downloaded from the authors' repository (it is not stored in ours).

One seed is 140 tweets x 3 calls = 420 planned calls. On the free tier this used about one day's quota per seed.

In [ ]:
RUN_API = False   # change to True to repeat the API runs

In [ ]:
if RUN_API:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    !test -d source_data || git clone -q --depth 1 https://github.com/alaa-a-a/Arabic-Twitter-corpus-for-flood-detection.git source_data
    !mkdir -p private_data && cp source_data/ArabicFloods.csv private_data/
    !python prepare_data.py --csv private_data/ArabicFloods.csv
    !python make_golden_set.py   # stops if the sample differs from m5/golden_set_ids.csv
else:
    print("Skipped (RUN_API is False).")

In [ ]:
SEED = 42   # one of 42, 123, 2026
if RUN_API:
    !python evaluate.py --cases private_data/golden_set.csv --out m5/runs/rerun/seed_{SEED}.csv --seed {SEED} --pause 10
else:
    print("Skipped (RUN_API is False).")

Same-seed check: rerun the cases whose labels differed across seeds, twice with seed 42, and compare
with each other and with the saved seed 42 run. This makes 3 calls per case per rerun.

In [ ]:
if RUN_API:
    import time
    from baseline import classify_tweet

    texts = pd.read_csv("private_data/golden_set.csv", dtype=str).set_index("ID")["tweet_text"]

    def rerun(seed):
        out = {}
        for cid in changed:
            r = classify_tweet(texts[cid], seed=seed)
            out[cid] = (r["relevance"], r["information_type"])
            time.sleep(5)
        return out

    first, second = rerun(42), rerun(42)
    saved = {cid: tuple(runs[42].loc[cid, keys]) for cid in changed}
    print(f"Rerun 1 matches the saved seed 42 run: {sum(first[c] == saved[c] for c in changed)} of {len(changed)}")
    print(f"Rerun 1 matches rerun 2: {sum(first[c] == second[c] for c in changed)} of {len(changed)}")
else:
    print("Skipped (RUN_API is False).")